# LAB-HW-04 — Clock、reset 与 physical I/O

**今天只解决一个问题：让一个有状态的 RTL block 使用真实 platform clock/reset，并通过 constraint 把 output 接到板上。**

前置：LAB-HW-03 已通过。今天仍然不启动 Linux、不学 AXI。

**Project Trace:** RMD-012A · T-HW-004/T-HW-011

## 1. 今天新增的三层连接

<svg xmlns="http://www.w3.org/2000/svg" width="860" height="310" viewBox="0 0 860 310" role="img" aria-label="LAB-HW-04 clock reset and physical output map">
  <rect x="30" y="35" width="175" height="95" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="118" y="68" text-anchor="middle" font-size="15">Zynq UltraScale+ PS</text>
  <text x="118" y="93" text-anchor="middle" font-size="12">pl_clk0</text>
  <text x="118" y="113" text-anchor="middle" font-size="12">pl_resetn0</text>
  <rect x="300" y="160" width="175" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="388" y="192" text-anchor="middle" font-size="15">proc_sys_reset</text>
  <text x="388" y="216" text-anchor="middle" font-size="12">peripheral_aresetn</text>
  <rect x="550" y="35" width="170" height="95" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="635" y="68" text-anchor="middle" font-size="15">kv260_blink_core</text>
  <text x="635" y="93" text-anchor="middle" font-size="12">counter / resetn</text>
  <text x="635" y="113" text-anchor="middle" font-size="12">bank45_gpio[4:0]</text>
  <rect x="550" y="205" width="170" height="65" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="635" y="233" text-anchor="middle" font-size="14">XDC / package pins</text>
  <text x="635" y="254" text-anchor="middle" font-size="12">J11 J10 K13 F11 A12</text>
  <rect x="760" y="205" width="80" height="65" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="800" y="233" text-anchor="middle" font-size="14">Bank45</text>
  <text x="800" y="254" text-anchor="middle" font-size="12">visible I/O</text>
  <path d="M205 78 L550 78" stroke="#333" stroke-width="2"/><polygon points="550,78 540,73 540,83" fill="#333"/>
  <text x="375" y="63" text-anchor="middle" font-size="12">pl_clk0</text>
  <path d="M205 112 L300 190" stroke="#333" stroke-width="2"/><polygon points="300,190 290,184 292,196" fill="#333"/>
  <text x="248" y="157" text-anchor="middle" font-size="12">pl_resetn0</text>
  <path d="M475 200 L585 130" stroke="#333" stroke-width="2"/><polygon points="585,130 573,131 579,140" fill="#333"/>
  <text x="535" y="180" text-anchor="middle" font-size="12">design-local resetn</text>
  <path d="M635 130 L635 205" stroke="#333" stroke-width="2"/><polygon points="635,205 630,195 640,195" fill="#333"/>
  <path d="M720 238 L760 238" stroke="#333" stroke-width="2"/><polygon points="760,238 750,233 750,243" fill="#333"/>
</svg>

这里第一次同时看到：

- **clock source**：PS `pl_clk0`；
- **reset source**：PS `pl_resetn0` → `proc_sys_reset`；
- **physical mapping**：`bank45_gpio[*]` → XDC → K26 package pin。

PS 今天只是 clock/reset provider，不承担 runtime software。

## 2. 有状态逻辑：为什么 marker 变成 blink

`kv260_blink_core` 内部是一个 26-bit counter。

nominal 100 MHz clock 下，最高位变化足够慢，可以肉眼观察。输出 contract：

- `bank45_gpio[0]` = counter 的最高位，周期变化；
- `bank45_gpio[4:1]` 保持 marker。

这一步把 Lesson 4 的“register 需要 clock”真正搬到 FPGA 上。

## 3. Reset 一定要分层

<svg xmlns="http://www.w3.org/2000/svg" width="820" height="170" viewBox="0 0 820 170" role="img" aria-label="LAB-HW-04 reset layers">
  <rect x="20" y="50" width="150" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="95" y="80" text-anchor="middle" font-size="15">PS pl_resetn0</text>
  <text x="95" y="102" text-anchor="middle" font-size="12">platform source</text>
  <rect x="235" y="50" width="170" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="320" y="80" text-anchor="middle" font-size="15">proc_sys_reset</text>
  <text x="320" y="102" text-anchor="middle" font-size="12">reset synchronizer</text>
  <rect x="470" y="50" width="165" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="552" y="80" text-anchor="middle" font-size="14">peripheral_aresetn</text>
  <text x="552" y="102" text-anchor="middle" font-size="12">design-local reset</text>
  <rect x="700" y="50" width="100" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="750" y="80" text-anchor="middle" font-size="14">blink core</text>
  <text x="750" y="102" text-anchor="middle" font-size="12">resetn</text>
  <path d="M170 85 L235 85 M405 85 L470 85 M635 85 L700 85" stroke="#333" stroke-width="2"/>
  <polygon points="235,85 225,80 225,90" fill="#333"/>
  <polygon points="470,85 460,80 460,90" fill="#333"/>
  <polygon points="700,85 690,80 690,90" fill="#333"/>
</svg>

`resetn` 是 active-low。

SW2 则是 SOM-level hard reset。按 SW2 可能在更上游导致整个系统重新进入 reset sequence，但：

**SW2 不是一根直接连接到 `kv260_blink_core.resetn` 的 RTL wire。**

如果这句话说不清楚，就还没有通过本 Lab 的 Human Check。

## 4. 现在再打开 XDC

打开：

`boards/kv260/constraints/bank45_gpio.xdc`

例如：

```tcl
set_property PACKAGE_PIN J11 [get_ports {bank45_gpio[0]}]
set_property IOSTANDARD LVCMOS33 [get_ports {bank45_gpio[0]}]
```

第一行回答：

> logical port 最后去 K26 package 的哪个 pin？

第二行回答：

> 这个 I/O bank 的电气 standard 是什么？

同一个 `bank45_gpio[0]` 在 SystemVerilog 中只是一个 bit；只有 constraint 才把它变成具体 physical I/O。

### Checkpoint A — 先只理解 physical mapping

运行 Vivado 之前，先只检查 **logical-port → physical-pin** 这一层：

1. 指出 XDC 中哪一行把 `bank45_gpio[0]` 映射到 package pin J11。
2. 解释为什么 `IOSTANDARD LVCMOS33` 是电气 constraint，而不是 SystemVerilog behavior。
3. 说明哪些事实来自课程 RTL，哪些来自 KV260 board mapping。

如果这三点还说不清楚，就停在这里；先不要把 clock/reset debugging 混进来。

## 5. Build：建立 PS clock/reset + PL core

从仓库根目录：

```bash
vivado -mode batch -nojournal \
  -log lab-hw-04-build.log \
  -source boards/kv260/scripts/build_lab04_blink.tcl
```

脚本会：

1. 建立 KV260/K26 Vivado project；
2. 使用 KV260 board preset 建立 Zynq UltraScale+ PS；
3. 建立 `proc_sys_reset`；
4. 把 `pl_clk0` / `pl_resetn0` 接到 reset controller 与 `kv260_blink_core`；
5. 生成 wrapper；
6. synthesis → implementation → bitstream；
7. 输出 timing/resource reports；
8. 如果没有 clock、缺少 setup/hold timing path、setup slack 为负或 hold slack 为负，则 build 直接 FAIL。

成功 artifact：

`build/kv260/lab-hw-04/kv260_blink.bit`

## 6. Program + Observe

先计算 SHA-256，然后复用 LAB-HW-03 的 programming helper：

```bash
vivado -mode batch -nojournal \
  -log lab-hw-04-program.log \
  -source boards/kv260/scripts/program_bitstream.tcl \
  -tclargs build/kv260/lab-hw-04/kv260_blink.bit
```

**Expected Evidence：**

- `xck26*` program success；
- Bank45 bit 0 的可见状态周期变化；
- 其他 marker bits 稳定；
- timing/resource report 存在；
- bitstream hash 已记录。

这次“变化”本身很重要：它证明不是一个静态上电状态，而是有 clock 驱动的 state evolution。

## 7. 如何理解 clock 的 timing evidence

build log 会记录 PS→PL clock/reset source，timing report 记录 implemented design 的 timing analysis。

本 Lab 不要求学习完整 clocking architecture；只需要知道：

- counter 的 register 是被真实 clock edge 驱动；
- clock 不是一个普通 data wire；
- timing analysis 必须知道 clock domain；
- 本 Lab 把 timing 当作 pass/fail oracle：worst setup slack 与 worst hold slack 都必须非负；
- 设计如果一直被 reset hold 住，LED 不会 blink。

build helper 会打印 `TIMING_SETUP_WORST_SLACK_NS` 与 `TIMING_HOLD_WORST_SLACK_NS`，任一为负就返回非零。不要看到 `program success` 就跳过 timing report。

### Checkpoint B — Program 之前单独检查 clock/reset/timing

build 完成后，再单独检查 **state-evolution** 这一层：

- Vivado 中真实存在 clock；
- setup 与 hold timing path 都存在；
- `TIMING_SETUP_WORST_SLACK_NS` 非负；
- `TIMING_HOLD_WORST_SLACK_NS` 非负；
- log 中 reset path 为 `pl_resetn0 -> proc_sys_reset/peripheral_aresetn -> blink_core/resetn`。

只有 Checkpoint A **和** Checkpoint B 都通过，才进入实体板 programming。

## 8. Save Evidence

T-HW-004 evidence 至少包括：

- `lab-hw-04-build.log`；
- `timing_summary.rpt`；
- `utilization.rpt`；
- `kv260_blink.bit` SHA-256；
- `lab-hw-04-program.log`；
- blink observation / photo/video；
- XDC 中至少一个 logical bit → package pin 的解释；
- carrier revision、Vivado version、Git commit。

继续用 `boards/kv260/evidence/manifest.example.json` 建立本地 evidence manifest。

## 9. If it does not work

按新加入的层级排查：

1. **LAB-HW-03 marker 都不能 program** → 不要继续，本 Lab 前置没过；
2. **board part / PS IP 建不起来** → 回到 LAB-HW-00 的 board definitions；
3. **synthesis/implementation fail** → 看 build log，不碰 JTAG；
4. **program success 但 output 完全不变化** → 检查 clock/reset connection，特别是 `pl_clk0` 与 `peripheral_aresetn`；
5. **只有部分物理 output 不符合** → 查 XDC / carrier revision / physical polarity；
6. **按 SW2 后行为改变** → 这是 system reset sequence 的 evidence，不要把它直接写成“我拉低了 RTL resetn”。

## 10. Human Check

1. `PACKAGE_PIN J11` 与 SystemVerilog 的 `bank45_gpio[0]` 分别属于哪一层？
2. 为什么 `IOSTANDARD LVCMOS33` 不是“代码风格”设置？
3. `pl_resetn0` 和 `peripheral_aresetn` 为什么不是同一个抽象层？
4. SW2 为什么不能被教材简写成“RTL reset button”？
5. 如果 program 成功但 counter 不动，你会先查 clock/reset 还是 neuron algorithm？为什么？
6. LAB-HW-04 比 LAB-HW-03 新增的主要概念是什么？

## 11. 官方依据

- AMD UG1089 — Vivado Board Flow  
  https://docs.amd.com/r/en-US/ug1089-kv260-starter-kit/Vivado-Board-Flow
- AMD UG1089 — Board Reset  
  https://docs.amd.com/r/en-US/ug1089-kv260-starter-kit/Board-Reset
- AMD/Xilinx Board Store — KV260 SOM board part 1.4 / K26 package mapping  
  https://github.com/Xilinx/XilinxBoardStore/tree/master/boards/Xilinx/kv260_som/1.4
- AMD/Xilinx examples use K26 `xck26-sfvc784-2LV-c`, KV260 board part, and PS `pl_clk0` for PL clocking.